# Regularisation and training-participant budgets

**Purpose:** Examine Ridge regularisation and eight-, twelve- and full-development-participant training budgets.

**Required inputs:** Prepared Baček HDF5; 04 results ZIP; E4_2_Proposed_Training_Participant_Subsets.csv.

**Outputs:** Model-selection, subset and learning-curve records.

**Usage:** Run after 04 using the fixed participant-subset manifest. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Imports, fixed definitions and reusable helpers
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import os
import platform
import sys
import time
import zipfile

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
REFERENCE = Path('04_Bacek_Foundational_Predictive_Models_results.zip')
SUBSETS = Path('E4_2_Proposed_Training_Participant_Subsets.csv')
OUT = Path('10_Bacek_Regularisation_Training_Participant_Sufficiency_results')
ARCHIVE = Path('10_Bacek_Regularisation_Training_Participant_Sufficiency_results.zip')
NOTEBOOK = Path('10_Bacek_Regularisation_Training_Participant_Sufficiency.ipynb')
SOURCE_SHA256 = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
SUBSET_SHA256 = 'b63a62267b99d63a9e649722f595d8a703f6bfb4d49227aca150139c122a8ef2'
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
CHANNELS = ('hip_left', 'hip_right', 'knee_left', 'knee_right',
            'ankle_left', 'ankle_right', 'pelvis_sagittal',
            'pelvis_frontal', 'pelvis_transverse')
OUTER_FOLDS = {
    1: ('S08', 'S10', 'S16', 'S21'),
    2: ('S11', 'S13', 'S17', 'S20'),
    3: ('S01', 'S02', 'S05', 'S18'),
    4: ('S03', 'S07', 'S12', 'S14', 'S19'),
    5: ('S04', 'S06', 'S09', 'S15'),
}
EXPECTED_COUNTS = (55, 53, 56, 65, 55)
FOLD_OF = {subject: fold for fold, people in OUTER_FOLDS.items() for subject in people}


def require(condition, message):
    if not bool(condition):
        raise AssertionError(message)


def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as file:
        for block in iter(lambda: file.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def csv_from_zip(archive, name):
    with archive.open(name) as file:
        return list(csv.DictReader(io.TextIOWrapper(file, encoding='utf-8', newline='')))


def csv_from_file(path):
    with path.open('r', newline='', encoding='utf-8-sig') as file:
        return list(csv.DictReader(file))


def write_csv(path, data, fields):
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(data)


def people(text):
    names = text.split(';')
    require(len(names) == len(set(names)) and all(names), 'Invalid or repeated participant IDs')
    return tuple(names)


def rmse(y_true, y_pred):
    error = np.asarray(y_pred, dtype=np.float64) - np.asarray(y_true, dtype=np.float64)
    return float(np.sqrt(np.mean(error**2, dtype=np.float64)))


def performance(y_true, y_pred):
    actual = np.asarray(y_true, dtype=np.float64)
    pred = np.asarray(y_pred, dtype=np.float64)
    require(len(actual) > 0 and len(actual) == len(pred), 'Empty or unequal evaluation vectors')
    error = pred - actual
    denominator = float(np.sum((actual-actual.mean())**2, dtype=np.float64))
    return {'n_trials': len(actual), 'RMSE_W_per_kg': rmse(actual, pred),
            'MAE_W_per_kg': float(np.mean(np.abs(error), dtype=np.float64)),
            'bias_W_per_kg': float(np.mean(error, dtype=np.float64)),
            'R2': float(1 - np.sum(error**2, dtype=np.float64)/denominator)
            if denominator > 0 else None}


def model(alpha):
    return Pipeline([('scale', StandardScaler()),
                     ('ridge', Ridge(alpha=float(alpha), fit_intercept=True))])


def choose_alpha(candidate_rows):
    return min(candidate_rows,
               key=lambda r: (r['inner_pooled_RMSE_W_per_kg'], float(r['alpha'])))


def near(a, b, tolerance=1e-10):
    return bool(np.isclose(float(a), float(b), atol=tolerance, rtol=0))

print('Kernel Python:', sys.executable)
print('Versions: Python', platform.python_version(), '| NumPy', np.__version__,
      '| scikit-learn', sklearn.__version__)


In [ ]:
# 2 — Validate Notebook 04 records and binding subset manifest BEFORE reading source arrays
require(H5.is_file() and REFERENCE.is_file() and SUBSETS.is_file(),
        'One or more required input files are missing from the notebook directory')
source_sha = digest(H5)
subset_sha = digest(SUBSETS)
reference_sha = digest(REFERENCE)
require(source_sha == SOURCE_SHA256, 'Corrected HDF5 source fingerprint mismatch')
require(subset_sha == SUBSET_SHA256, 'E4.2 fixed subset manifest fingerprint mismatch')

with zipfile.ZipFile(REFERENCE) as zip04:
    require(zip04.testzip() is None, 'Notebook 04 results ZIP contains a corrupt member')
    ref_summary = json.loads(zip04.read('run_summary.json').decode('utf-8'))
    ref_ledger = csv_from_zip(zip04, 'eligible_trial_ledger.csv')
    ref_outer = csv_from_zip(zip04, 'outer_fold_manifest.csv')
    ref_inner = csv_from_zip(zip04, 'inner_fold_manifest.csv')
    ref_inner_scores = csv_from_zip(zip04, 'inner_candidate_scores.csv')
    ref_pooled_scores = csv_from_zip(zip04, 'pooled_candidate_scores.csv')
    ref_selected = csv_from_zip(zip04, 'selected_models.csv')
    ref_predictions = csv_from_zip(zip04, 'held_out_predictions.csv')
    ref_timings = csv_from_zip(zip04, 'outer_model_timings.csv')

require(ref_summary['status'] == 'PASS' and ref_summary['source_sha256'] == source_sha,
        'Notebook 04 provenance or verification does not match')
require(ref_summary['alpha_grid'] == list(ALPHAS), 'Notebook 04 Ridge grid differs')
require(len(ref_ledger) == 284 and len(ref_outer) == 5 and len(ref_inner) == 20,
        'Notebook 04 population/fold record count differs')
require(len(ref_inner_scores) == 960 and len(ref_pooled_scores) == 240,
        'Notebook 04 candidate score coverage differs')
require(len(ref_selected) == 30 and len(ref_predictions) == 7 * 284,
        'Notebook 04 selection or prediction count differs')
require(len({r['trial_uid'] for r in ref_ledger}) == 284, 'Repeated trial keys in reference')
require(set(r['subject_id'] for r in ref_ledger) == set(FOLD_OF),
        'Reference participant population differs')
require(sum(int(r['retained_cycles']) for r in ref_ledger) == 60757,
        'Reference retained-cycle total differs')
for fold, heldout in OUTER_FOLDS.items():
    outer_row = next(r for r in ref_outer if int(r['outer_fold']) == fold)
    development = set(FOLD_OF) - set(heldout)
    require(set(people(outer_row['test_participants'])) == set(heldout),
            'Reference outer-test participants differ')
    require(set(people(outer_row['development_participants'])) == development,
            'Reference outer-development participants differ')
    require(sum(int(r['outer_fold']) == fold for r in ref_ledger) == EXPECTED_COUNTS[fold-1],
            'Reference outer-test trial count differs')
    inner_rows = [r for r in ref_inner if int(r['outer_fold']) == fold]
    require(len(inner_rows) == 4, 'Missing full-development inner folds')
    validation_groups = []
    for row in inner_rows:
        train, validation = set(people(row['train_participants'])), set(people(row['validation_participants']))
        require(train.isdisjoint(validation) and train | validation == development,
                'Reference full-development inner groups do not partition development people')
        validation_groups.extend(validation)
    require(sorted(validation_groups) == sorted(development),
            'Reference inner folds do not cover development participants once')
for row in ref_ledger:
    require(FOLD_OF[row['subject_id']] == int(row['outer_fold']),
            'Trial assigned to the wrong outer participant fold')

subset_rows = csv_from_file(SUBSETS)
require(len(subset_rows) == 55, 'E4.2 manifest must contain exactly 55 rows')
manifest = {}
for row in subset_rows:
    fold = int(row['outer_fold'])
    budget = int(row['training_budget'])
    repeat = row['subset_replicate']
    ids = people(row['training_participants'])
    outer = people(row['outer_test_participants'])
    require(fold in OUTER_FOLDS and set(outer) == set(OUTER_FOLDS[fold]),
            'Subset outer-test participants differ from frozen groups')
    require(ids == tuple(sorted(ids)), 'Subset participant IDs are not canonical/sorted')
    require(set(ids).issubset(set(FOLD_OF) - set(outer)),
            'Subset contains an outer-test or unknown participant')
    require(len(ids) == budget, 'Subset size does not match budget')
    key = (fold, budget, repeat)
    require(key not in manifest, 'Repeated manifest subset key')
    manifest[key] = tuple(ids)
for fold, outer in OUTER_FOLDS.items():
    development = set(FOLD_OF) - set(outer)
    full_budget = len(development)
    require(manifest.get((fold, full_budget, 'full')) is not None and
            set(manifest[(fold, full_budget, 'full')]) == development,
            'Full-development subset does not equal the outer-development group')
    for repeat in range(1, 6):
        eight = manifest.get((fold, 8, str(repeat)))
        twelve = manifest.get((fold, 12, str(repeat)))
        require(eight is not None and twelve is not None and set(eight).issubset(twelve),
                'Each budget-8 subset must be nested in budget 12 for the same repeat')
require(len(manifest) == 55, 'Missing or unexpected subset definition')

# Extract full-development D-ALL9 data for exact reuse.
full_candidates = {(int(r['outer_fold']), float(r['alpha'])): r for r in ref_pooled_scores
                   if r['representation'] == 'D-ALL9'}
full_choices = {int(r['outer_fold']): r for r in ref_selected
                if r['procedure'] == 'D-ALL9_FIXED'}
full_predictions = {(int(r['outer_fold']), r['trial_uid']): r for r in ref_predictions
                    if r['procedure'] == 'D-ALL9_FIXED'}
require(len(full_candidates) == 30 and len(full_choices) == 5 and len(full_predictions) == 284,
        'Incomplete D-ALL9 reference records')
for fold in OUTER_FOLDS:
    scores = [full_candidates[(fold, alpha)] for alpha in ALPHAS]
    selected = choose_alpha([{'alpha': alpha,
                              'inner_pooled_RMSE_W_per_kg': float(full_candidates[(fold, alpha)]['inner_pooled_RMSE_W_per_kg'])}
                             for alpha in ALPHAS])
    require(near(float(full_choices[fold]['alpha']), selected['alpha'], 0) and
            near(full_choices[fold]['inner_pooled_RMSE_W_per_kg'],
                 selected['inner_pooled_RMSE_W_per_kg']),
            'Full-development reference selection mismatch')

print('Input fingerprints and fixed subsets verified:', len(manifest), 'definitions')
print('Notebook 04 full-development D-ALL9 reference:', len(full_predictions), 'predictions')


In [ ]:
# 3 — Reuse W/D regularisation development records, including training and validation RMSE
regularisation_inner = []
regularisation_summary = []
for fold in OUTER_FOLDS:
    for representation in ('W-ALL9', 'D-ALL9'):
        for alpha in ALPHAS:
            records = [r for r in ref_inner_scores
                       if int(r['outer_fold']) == fold and
                       r['representation'] == representation and float(r['alpha']) == alpha]
            require(len(records) == 4 and {int(r['inner_fold']) for r in records} == {1, 2, 3, 4},
                    'Missing regularisation inner-fold score')
            numerator = 0.0
            denominator = 0
            for item in records:
                n = int(item['n_validation_trials'])
                val_rmse = float(item['validation_RMSE_W_per_kg'])
                numerator += n * val_rmse**2
                denominator += n
                regularisation_inner.append({
                    'outer_fold': fold, 'representation': representation, 'alpha': alpha,
                    'inner_fold': int(item['inner_fold']),
                    'n_train_trials': int(item['n_train_trials']),
                    'n_validation_trials': n,
                    'train_RMSE_W_per_kg': float(item['train_RMSE_W_per_kg']),
                    'validation_RMSE_W_per_kg': val_rmse,
                    'source': 'reused_notebook_04',
                })
            pooled = float(np.sqrt(numerator / denominator))
            stored = next(r for r in ref_pooled_scores
                          if int(r['outer_fold']) == fold and
                          r['representation'] == representation and float(r['alpha']) == alpha)
            require(near(pooled, stored['inner_pooled_RMSE_W_per_kg']),
                    'Reference pooled validation RMSE does not match fold-level scores')
            selected_procedure = 'W-ALL9_FIXED' if representation == 'W-ALL9' else 'D-ALL9_FIXED'
            selected_row = next(r for r in ref_selected
                                if int(r['outer_fold']) == fold and r['procedure'] == selected_procedure)
            regularisation_summary.append({
                'outer_fold': fold, 'representation': representation, 'alpha': alpha,
                'mean_inner_training_RMSE_W_per_kg_descriptive': float(np.mean(
                    [float(r['train_RMSE_W_per_kg']) for r in records], dtype=np.float64)),
                'inner_pooled_validation_RMSE_W_per_kg': pooled,
                'is_inner_selected_alpha': float(selected_row['alpha']) == alpha,
                'source': 'reused_notebook_04',
            })
require(len(regularisation_inner) == 240 and len(regularisation_summary) == 60,
        'Regularisation record count differs')
print('Regularisation evidence reused:', len(regularisation_inner), 'inner-fold records,',
      len(regularisation_summary), 'per-fold alpha summaries')
print('Training RMSE columns are descriptive in-sample values; only pooled validation RMSE is used for selection.')


In [ ]:
# 4 — Small synthetic verification of the fixed D mathematics and pipeline behaviour
phase = np.arange(101, dtype=np.float64)
synthetic_wave = np.tile(phase, (9, 1))

def descriptors_from_means(means):
    require(means.shape == (9, 101) and np.isfinite(means).all(),
            'Invalid nine-channel trial mean input')
    descriptors = []
    for channel in means:
        for q in range(4):
            descriptors.append(float(np.trapezoid(channel[q*25:q*25+26], dx=1.0) / 25.0))
        descriptors.append(float(np.std(channel, ddof=0)))
        descriptors.append(float(np.max(channel) - np.min(channel)))
    return np.asarray(descriptors, dtype=np.float64)

synthetic_D = descriptors_from_means(synthetic_wave)
require(synthetic_D.shape == (54,), 'Synthetic descriptor dimensions differ')
require(np.allclose(synthetic_D[:4], [12.5, 37.5, 62.5, 87.5], atol=1e-12),
        'Synthetic quartile integration failed')
require(np.isclose(synthetic_D[4], np.std(phase, ddof=0)) and
        synthetic_D[5] == 100.0, 'Synthetic spread descriptors failed')
# Check the smallest-alpha tie rule and fitting entirely inside a supplied training partition.
require(choose_alpha([{'alpha': 1.0, 'inner_pooled_RMSE_W_per_kg': 0.5},
                      {'alpha': 0.1, 'inner_pooled_RMSE_W_per_kg': 0.5}])['alpha'] == 0.1,
        'Synthetic exact-tie rule failed')
test_X = np.array([[0.], [2.], [4.], [100.]], dtype=np.float64)
test_y = np.array([1., 2., 3., 4.])
test_pipeline = model(1.0).fit(test_X[:3], test_y[:3])
require(near(test_pipeline.named_steps['scale'].mean_[0], 2.0, 0),
        'Scaler must use training inputs only')
require(np.isfinite(test_pipeline.predict(test_X[3:])).all(), 'Synthetic prediction failed')
print('Synthetic descriptors, exact ties and training-only scaling: PASS')


In [ ]:
# 5 — Read the actual corrected HDF5 locally; reconstruct D-ALL9 and match Notebook 04 exactly
trial_rows = sorted(ref_ledger, key=lambda r: r['trial_uid'])
require(len(trial_rows) == 284 and len(set(r['trial_uid'] for r in trial_rows)) == 284,
        'Invalid primary trial ledger')
Xd_rows = []
with h5py.File(H5, 'r') as h5:
    require(sorted(h5['subjects']) == sorted(FOLD_OF), 'Source participants differ')
    for row in trial_rows:
        subject, trial_id = row['subject_id'], row['trial_id']
        trial = h5['subjects'][subject]['trials'][trial_id]
        uid = row['trial_uid']
        require(uid == f'{subject}_{trial_id}', 'Trial key mismatch')
        for attribute in ('valid', 'comparison_eligible', 'unrestricted_comparison_eligible'):
            require(attribute in trial.attrs and
                    np.asarray(trial.attrs[attribute]).reshape(-1)[0] == 1,
                    f'{uid}: invalid status {attribute}')
        mean_waveforms = []
        n_cycles = None
        for channel in CHANNELS:
            require(channel in trial, f'{uid}: missing {channel}')
            values = np.asarray(trial[channel][()], dtype=np.float64)
            require(values.ndim == 2 and values.shape[1] == 101 and
                    np.isfinite(values).all(), f'{uid}: invalid {channel} shape/values')
            if n_cycles is None:
                n_cycles = values.shape[0]
            require(values.shape[0] == n_cycles, f'{uid}: cycle counts differ')
            mean_waveforms.append(values.mean(axis=0, dtype=np.float64))
        require(n_cycles == int(row['retained_cycles']) and n_cycles >= 100,
                f'{uid}: retained-cycle count mismatch')
        require('original_cycle_number' in trial, f'{uid}: missing original cycle numbers')
        indices = np.asarray(trial['original_cycle_number'][()]).reshape(-1)
        require(len(indices) == n_cycles and np.isfinite(indices).all() and
                np.all(indices == np.floor(indices)) and np.all(np.diff(indices) > 0),
                f'{uid}: invalid retained-cycle provenance')
        require('metcost_mean' in trial, f'{uid}: missing corrected scalar target')
        target = np.asarray(trial['metcost_mean'][()])
        require(target.size == 1 and np.issubdtype(target.dtype, np.number) and
                np.isrealobj(target), f'{uid}: invalid stored target')
        value = float(target.reshape(-1)[0])
        require(np.isfinite(value) and value > 0 and
                value == float(row['observed_W_per_kg']),
                f'{uid}: target does not match Notebook 04')
        require(FOLD_OF[subject] == int(row['outer_fold']),
                f'{uid}: outer-fold assignment changed')
        Xd_rows.append(descriptors_from_means(np.stack(mean_waveforms)))

X = np.stack(Xd_rows)
y = np.array([float(r['observed_W_per_kg']) for r in trial_rows], dtype=np.float64)
groups = np.array([r['subject_id'] for r in trial_rows])
uids = np.array([r['trial_uid'] for r in trial_rows])
require(X.shape == (284, 54) and np.isfinite(X).all(), 'D-ALL9 matrix invalid')
require(len(set(groups)) == 21 and len(set(uids)) == 284, 'Population coverage differs')
uid_to_index = {uid: i for i, uid in enumerate(uids)}
for key, row in full_predictions.items():
    require(row['trial_uid'] in uid_to_index and
            near(row['observed_W_per_kg'], y[uid_to_index[row['trial_uid']]], 0),
            'Reference D-ALL9 evaluation target differs')
print('D-ALL9 constructed from verified local HDF5:', X.shape)


## Nested training-participant subset evaluation


In [ ]:
# 6 — Execute E4.2 participant-budget models and record complete development evidence
started_at = datetime.now(timezone.utc).isoformat()
t0_all = time.perf_counter()
subset_inner_manifest = []
subset_inner_scores = []
subset_candidate_scores = []
subset_selected = []
subset_predictions = []
subset_timings = []

for fold, test_people in OUTER_FOLDS.items():
    test_idx = np.flatnonzero(np.isin(groups, test_people))
    development = set(FOLD_OF) - set(test_people)
    require(len(test_idx) == EXPECTED_COUNTS[fold-1], 'Outer trial population differs')
    require(set(groups[test_idx]) == set(test_people), 'Outer-test participant coverage differs')

    # Five distinct, fixed nested pairs of 8/12-person training subsets.
    for repeat in range(1, 6):
        for budget in (8, 12):
            selected_people = manifest[(fold, budget, str(repeat))]
            train_idx = np.flatnonzero(np.isin(groups, selected_people))
            require(set(groups[train_idx]) == set(selected_people) and
                    set(groups[train_idx]).isdisjoint(test_people),
                    'Subset coverage or outer-test separation differs')
            require(len(train_idx) > budget and set(groups[train_idx]).issubset(development),
                    'Invalid subset trial population')

            inner_splits = []
            covered = set()
            for inner_fold, (local_train, local_val) in enumerate(
                    GroupKFold(n_splits=4).split(X[train_idx], y[train_idx], groups[train_idx]), 1):
                fit_idx, val_idx = train_idx[local_train], train_idx[local_val]
                train_names = set(groups[fit_idx])
                val_names = set(groups[val_idx])
                require(train_names.isdisjoint(val_names) and
                        train_names | val_names == set(selected_people),
                        'Participant leakage in subset inner cross-validation')
                require(covered.isdisjoint(val_idx.tolist()), 'Repeated inner-validation trials')
                covered.update(val_idx.tolist())
                inner_splits.append((inner_fold, fit_idx, val_idx))
                subset_inner_manifest.append({
                    'outer_fold': fold, 'budget_group': str(budget), 'training_budget': budget,
                    'subset_replicate': repeat, 'inner_fold': inner_fold,
                    'train_participants': ';'.join(sorted(train_names)),
                    'validation_participants': ';'.join(sorted(val_names)),
                    'outer_test_participants': ';'.join(test_people),
                    'n_train_trials': len(fit_idx), 'n_validation_trials': len(val_idx),
                    'source': 'new_subset_GroupKFold',
                })
            require(covered == set(train_idx.tolist()), 'Incomplete inner-validation coverage')

            trial_position = {int(index): position for position, index in enumerate(train_idx)}
            candidates = []
            for alpha in ALPHAS:
                pooled = np.full(len(train_idx), np.nan, dtype=np.float64)
                for inner_fold, fit_idx, val_idx in inner_splits:
                    estimator = model(alpha)
                    t0 = time.perf_counter()
                    estimator.fit(X[fit_idx], y[fit_idx])
                    fit_seconds = time.perf_counter() - t0
                    p_train = np.asarray(estimator.predict(X[fit_idx]), dtype=np.float64)
                    p_val = np.asarray(estimator.predict(X[val_idx]), dtype=np.float64)
                    require(np.isfinite(p_train).all() and np.isfinite(p_val).all(),
                            'Nonfinite inner predictions')
                    for index, predicted in zip(val_idx, p_val):
                        pooled[trial_position[int(index)]] = predicted
                    subset_inner_scores.append({
                        'outer_fold': fold, 'budget_group': str(budget),
                        'training_budget': budget, 'subset_replicate': repeat,
                        'inner_fold': inner_fold, 'representation': 'D-ALL9',
                        'alpha': alpha, 'n_train_trials': len(fit_idx),
                        'n_validation_trials': len(val_idx),
                        'train_RMSE_W_per_kg': rmse(y[fit_idx], p_train),
                        'validation_RMSE_W_per_kg': rmse(y[val_idx], p_val),
                        'fit_seconds': fit_seconds, 'source': 'new_subset_fit',
                    })
                require(np.isfinite(pooled).all(), 'Candidate pooled inner predictions incomplete')
                candidate = {'outer_fold': fold, 'budget_group': str(budget),
                             'training_budget': budget, 'subset_replicate': repeat,
                             'representation': 'D-ALL9', 'alpha': alpha,
                             'inner_pooled_RMSE_W_per_kg': rmse(y[train_idx], pooled),
                             'source': 'new_subset_fit'}
                subset_candidate_scores.append(candidate)
                candidates.append(candidate)
            chosen = choose_alpha(candidates)
            subset_selected.append(dict(chosen, n_train_trials=len(train_idx),
                                        n_training_participants=budget))
            final_model = model(chosen['alpha'])
            t0 = time.perf_counter()
            final_model.fit(X[train_idx], y[train_idx])
            fit_seconds = time.perf_counter() - t0
            t1 = time.perf_counter()
            predictions = np.asarray(final_model.predict(X[test_idx]), dtype=np.float64)
            prediction_seconds = time.perf_counter() - t1
            require(np.isfinite(predictions).all(), 'Nonfinite outer-test predictions')
            subset_timings.append({'outer_fold': fold, 'budget_group': str(budget),
                                   'training_budget': budget, 'subset_replicate': repeat,
                                   'fit_seconds': fit_seconds, 'prediction_seconds': prediction_seconds,
                                   'n_training_trials': len(train_idx),
                                   'n_predicted_trials': len(test_idx), 'source': 'new_subset_fit'})
            for position, index in enumerate(test_idx):
                subset_predictions.append({
                    'outer_fold': fold, 'budget_group': str(budget),
                    'training_budget': budget, 'subset_replicate': repeat,
                    'subject_id': groups[index], 'trial_uid': uids[index],
                    'observed_W_per_kg': float(y[index]),
                    'predicted_W_per_kg': float(predictions[position]),
                    'source': 'new_subset_fit',
                })
    # Full development: copy its candidate scores, inner-fold scores and predictions.
    full_budget = len(development)
    full_key = {'outer_fold': fold, 'budget_group': 'full',
                'training_budget': full_budget, 'subset_replicate': 'full'}
    for reference in ref_inner:
        if int(reference['outer_fold']) == fold:
            subset_inner_manifest.append({**full_key, 'inner_fold': int(reference['inner_fold']),
                'train_participants': reference['train_participants'],
                'validation_participants': reference['validation_participants'],
                'outer_test_participants': reference['outer_test_participants'],
                'n_train_trials': int(reference['n_train_trials']),
                'n_validation_trials': int(reference['n_validation_trials']),
                'source': 'reused_notebook_04'})
    for reference in ref_inner_scores:
        if int(reference['outer_fold']) == fold and reference['representation'] == 'D-ALL9':
            subset_inner_scores.append({**full_key, 'inner_fold': int(reference['inner_fold']),
                'representation': 'D-ALL9', 'alpha': float(reference['alpha']),
                'n_train_trials': int(reference['n_train_trials']),
                'n_validation_trials': int(reference['n_validation_trials']),
                'train_RMSE_W_per_kg': float(reference['train_RMSE_W_per_kg']),
                'validation_RMSE_W_per_kg': float(reference['validation_RMSE_W_per_kg']),
                'fit_seconds': float(reference['fit_seconds']), 'source': 'reused_notebook_04'})
    for alpha in ALPHAS:
        reference = full_candidates[(fold, alpha)]
        subset_candidate_scores.append({**full_key, 'representation': 'D-ALL9',
            'alpha': alpha,
            'inner_pooled_RMSE_W_per_kg': float(reference['inner_pooled_RMSE_W_per_kg']),
            'source': 'reused_notebook_04'})
    chosen = full_choices[fold]
    subset_selected.append({**full_key, 'representation': 'D-ALL9',
        'alpha': float(chosen['alpha']),
        'inner_pooled_RMSE_W_per_kg': float(chosen['inner_pooled_RMSE_W_per_kg']),
        'source': 'reused_notebook_04',
        'n_train_trials': int(next(r for r in ref_outer if int(r['outer_fold']) == fold)['n_development_trials']),
        'n_training_participants': full_budget})
    for index in test_idx:
        reference = full_predictions[(fold, uids[index])]
        subset_predictions.append({**full_key, 'subject_id': groups[index],
            'trial_uid': uids[index], 'observed_W_per_kg': float(y[index]),
            'predicted_W_per_kg': float(reference['predicted_W_per_kg']),
            'source': 'reused_notebook_04'})
    timing = [r for r in ref_timings if int(r['outer_fold']) == fold and
              r['representation'] == 'D-ALL9' and float(r['alpha']) == float(chosen['alpha'])]
    require(len(timing) == 1, 'Full-development D-ALL9 timing reference missing')
    subset_timings.append({**full_key, 'fit_seconds': float(timing[0]['fit_seconds']),
                           'prediction_seconds': float(timing[0]['prediction_seconds']),
                           'n_training_trials': int(timing[0]['n_train_trials']),
                           'n_predicted_trials': int(timing[0]['n_predicted_trials']),
                           'source': 'reused_notebook_04'})
    print(f'Outer fold {fold}: 10 reduced-subset fits + one reused full-development procedure')

elapsed_seconds = time.perf_counter() - t0_all
print(f'Completed {len(subset_selected)} subset procedures in {elapsed_seconds:.1f} s')


In [ ]:
# 7 — Independently check coverage, candidate selections and calculate matched outcomes
checks = []

def verify(label, condition):
    require(condition, label)
    checks.append({'check': label, 'status': 'PASS'})

verify('input_source_hdf5_checksum', source_sha == SOURCE_SHA256)
verify('binding_subset_manifest_checksum', subset_sha == SUBSET_SHA256)
verify('reference_notebook_04_status', ref_summary['status'] == 'PASS')
verify('reference_trial_population_284', len(trial_rows) == 284)
verify('reference_primary_cycle_total', sum(int(r['retained_cycles']) for r in trial_rows) == 60757)
verify('manifest_55_definitions', len(manifest) == 55)
verify('descriptor_feature_matrix_dimensions', X.shape == (284, 54))
verify('regularisation_inner_records', len(regularisation_inner) == 240)
verify('regularisation_summary_records', len(regularisation_summary) == 60)
verify('inner_fold_records_55_times_4', len(subset_inner_manifest) == 220)
verify('inner_candidate_records_55_times_6_times_4', len(subset_inner_scores) == 1320)
verify('pooled_candidate_records_55_times_6', len(subset_candidate_scores) == 330)
verify('selected_models_55', len(subset_selected) == 55)
verify('expected_prediction_count', len(subset_predictions) == 284 * 11)
verify('all_predictions_finite', all(np.isfinite(r['predicted_W_per_kg']) for r in subset_predictions))
verify('unique_prediction_keys', len({(r['budget_group'], str(r['subset_replicate']), r['trial_uid'])
                                     for r in subset_predictions}) == len(subset_predictions))

for fold in OUTER_FOLDS:
    for repeat in ['1', '2', '3', '4', '5', 'full']:
        budgets = [8, 12] if repeat != 'full' else [len(set(FOLD_OF) - set(OUTER_FOLDS[fold]))]
        for budget in budgets:
            budget_group = str(budget) if repeat != 'full' else 'full'
            selection = [r for r in subset_selected if r['outer_fold'] == fold and
                         r['budget_group'] == budget_group and str(r['subset_replicate']) == repeat]
            candidates = [r for r in subset_candidate_scores if r['outer_fold'] == fold and
                          r['budget_group'] == budget_group and str(r['subset_replicate']) == repeat]
            require(len(selection) == 1 and len(candidates) == 6,
                    'Subset has incomplete selection records')
            winner = choose_alpha(candidates)
            require(selection[0]['alpha'] == winner['alpha'] and
                    near(selection[0]['inner_pooled_RMSE_W_per_kg'],
                         winner['inner_pooled_RMSE_W_per_kg']),
                    'Selected alpha differs from minimum pooled inner RMSE')
            scored_inner = [r for r in subset_inner_scores if r['outer_fold'] == fold and
                            r['budget_group'] == budget_group and str(r['subset_replicate']) == repeat]
            require(len(scored_inner) == 24, 'Missing 4×6 inner candidate records')
            for candidate in candidates:
                rows_c = [r for r in scored_inner if r['alpha'] == candidate['alpha']]
                require(len(rows_c) == 4, 'Missing inner-fold candidate records')
                n = sum(r['n_validation_trials'] for r in rows_c)
                pooled = float(np.sqrt(sum(r['n_validation_trials'] *
                                           r['validation_RMSE_W_per_kg']**2 for r in rows_c) / n))
                require(near(pooled, candidate['inner_pooled_RMSE_W_per_kg']),
                        'Pooled candidate RMSE not reproducible from inner folds')

for budget_group, repeat in [(str(b), n) for b in (8, 12) for n in range(1, 6)] + [('full', 'full')]:
    selected = [r for r in subset_predictions if r['budget_group'] == budget_group and
                str(r['subset_replicate']) == str(repeat)]
    require(len(selected) == 284 and {r['trial_uid'] for r in selected} == set(uids),
            'Budget/repeat does not have exactly one prediction per eligible trial')
    for row in selected:
        idx = uid_to_index[row['trial_uid']]
        require(row['outer_fold'] == FOLD_OF[groups[idx]] and
                row['observed_W_per_kg'] == float(y[idx]),
                'Unmatched evaluation trial or target')

verify('all_grouped_selections_and_trial_matches', True)
verify('full_development_predictions_exactly_reused', all(
    float(row['predicted_W_per_kg']) ==
    float(full_predictions[(row['outer_fold'], row['trial_uid'])]['predicted_W_per_kg'])
    for row in subset_predictions if row['budget_group'] == 'full'))
verify('full_development_selection_exactly_reused', all(
    row['alpha'] == float(full_choices[row['outer_fold']]['alpha'])
    for row in subset_selected if row['budget_group'] == 'full'))
verify('full_development_train_participant_count', sum(r['budget_group'] == 'full'
                                                        for r in subset_selected) == 5)

curve_metrics = []
curve_fold_metrics = []
for budget_group, repeat in [(str(b), n) for b in (8, 12) for n in range(1, 6)] + [('full', 'full')]:
    subset = [r for r in subset_predictions if r['budget_group'] == budget_group and
              str(r['subset_replicate']) == str(repeat)]
    curve_metrics.append({'budget_group': budget_group, 'subset_replicate': repeat,
                          'n_training_participants_per_outer_fold': budget_group if budget_group != 'full' else '16-17',
                          **performance([r['observed_W_per_kg'] for r in subset],
                                        [r['predicted_W_per_kg'] for r in subset])})
    for fold in OUTER_FOLDS:
        chosen = [r for r in subset if r['outer_fold'] == fold]
        curve_fold_metrics.append({'outer_fold': fold, 'budget_group': budget_group,
                                   'subset_replicate': repeat,
                                   **performance([r['observed_W_per_kg'] for r in chosen],
                                                 [r['predicted_W_per_kg'] for r in chosen])})
verify('pooled_11_metrics_and_55_outer_metrics', len(curve_metrics) == 11 and
       len(curve_fold_metrics) == 55)
full_rmse = next(r['RMSE_W_per_kg'] for r in curve_metrics if r['budget_group'] == 'full')
contrasts = []
for repeat in range(1, 6):
    eight = next(r['RMSE_W_per_kg'] for r in curve_metrics
                 if r['budget_group'] == '8' and r['subset_replicate'] == repeat)
    twelve = next(r['RMSE_W_per_kg'] for r in curve_metrics
                  if r['budget_group'] == '12' and r['subset_replicate'] == repeat)
    contrasts.append({'subset_replicate': repeat,
                      'rmse_budget8_W_per_kg': eight,
                      'rmse_budget12_W_per_kg': twelve,
                      'rmse_full_W_per_kg': full_rmse,
                      'budget8_minus_budget12_RMSE_W_per_kg': eight - twelve,
                      'budget8_minus_full_RMSE_W_per_kg': eight - full_rmse,
                      'budget12_minus_full_RMSE_W_per_kg': twelve - full_rmse})
verify('five_nested_budget_contrasts', len(contrasts) == 5)
verify('all_recorded_checks_pass', all(r['status'] == 'PASS' for r in checks))

print('\nE4.2 pooled learning-curve RMSE (W/kg), descriptive:')
for r in sorted(curve_metrics, key=lambda item: (item['budget_group'] == 'full',
                                                  int(item['budget_group']) if item['budget_group'] != 'full' else 999,
                                                  str(item['subset_replicate']))):
    print(f"  {r['budget_group']:>4s} people / repeat {str(r['subset_replicate']):>4s}: "
          f"{r['RMSE_W_per_kg']:.6f}")
print('\nRecorded checks:', len(checks), 'PASS')


In [ ]:
# 8 — Export source-linked evidence and ZIP archive
OUT.mkdir(exist_ok=True, parents=True)
notebook_code_sha = None
if NOTEBOOK.is_file():
    notebook_object = json.loads(NOTEBOOK.read_text(encoding='utf-8'))
    source_code = ''.join(''.join(c['source']) for c in notebook_object['cells']
                          if c['cell_type'] == 'code')
    notebook_code_sha = hashlib.sha256(source_code.encode('utf-8')).hexdigest()

specifications = [
    ('eligible_trial_ledger.csv', trial_rows,
     ['trial_uid', 'subject_id', 'trial_id', 'outer_fold', 'retained_cycles',
      'speed_m_s', 'relative_cadence', 'condition_index', 'speed_index', 'observed_W_per_kg']),
    ('training_subset_manifest_verified.csv', subset_rows,
     ['outer_fold', 'training_budget', 'subset_replicate',
      'training_participants', 'outer_test_participants']),
    ('regularisation_inner_scores.csv', regularisation_inner,
     ['outer_fold', 'representation', 'alpha', 'inner_fold', 'n_train_trials',
      'n_validation_trials', 'train_RMSE_W_per_kg', 'validation_RMSE_W_per_kg', 'source']),
    ('regularisation_summary.csv', regularisation_summary,
     ['outer_fold', 'representation', 'alpha',
      'mean_inner_training_RMSE_W_per_kg_descriptive',
      'inner_pooled_validation_RMSE_W_per_kg', 'is_inner_selected_alpha', 'source']),
    ('learning_curve_inner_fold_manifest.csv', subset_inner_manifest,
     ['outer_fold', 'budget_group', 'training_budget', 'subset_replicate',
      'inner_fold', 'train_participants', 'validation_participants',
      'outer_test_participants', 'n_train_trials', 'n_validation_trials', 'source']),
    ('learning_curve_inner_candidate_scores.csv', subset_inner_scores,
     ['outer_fold', 'budget_group', 'training_budget', 'subset_replicate',
      'inner_fold', 'representation', 'alpha', 'n_train_trials',
      'n_validation_trials', 'train_RMSE_W_per_kg',
      'validation_RMSE_W_per_kg', 'fit_seconds', 'source']),
    ('learning_curve_pooled_candidate_scores.csv', subset_candidate_scores,
     ['outer_fold', 'budget_group', 'training_budget', 'subset_replicate',
      'representation', 'alpha', 'inner_pooled_RMSE_W_per_kg', 'source']),
    ('learning_curve_selected_models.csv', subset_selected,
     ['outer_fold', 'budget_group', 'training_budget', 'subset_replicate',
      'representation', 'alpha', 'inner_pooled_RMSE_W_per_kg', 'source',
      'n_train_trials', 'n_training_participants']),
    ('learning_curve_predictions.csv', subset_predictions,
     ['outer_fold', 'budget_group', 'training_budget', 'subset_replicate',
      'subject_id', 'trial_uid', 'observed_W_per_kg', 'predicted_W_per_kg', 'source']),
    ('learning_curve_metrics.csv', curve_metrics,
     ['budget_group', 'subset_replicate', 'n_training_participants_per_outer_fold',
      'n_trials', 'RMSE_W_per_kg', 'MAE_W_per_kg', 'bias_W_per_kg', 'R2']),
    ('learning_curve_outer_fold_metrics.csv', curve_fold_metrics,
     ['outer_fold', 'budget_group', 'subset_replicate', 'n_trials',
      'RMSE_W_per_kg', 'MAE_W_per_kg', 'bias_W_per_kg', 'R2']),
    ('learning_curve_matched_contrasts.csv', contrasts,
     ['subset_replicate', 'rmse_budget8_W_per_kg', 'rmse_budget12_W_per_kg',
      'rmse_full_W_per_kg', 'budget8_minus_budget12_RMSE_W_per_kg',
      'budget8_minus_full_RMSE_W_per_kg', 'budget12_minus_full_RMSE_W_per_kg']),
    ('learning_curve_model_timings.csv', subset_timings,
     ['outer_fold', 'budget_group', 'training_budget', 'subset_replicate',
      'fit_seconds', 'prediction_seconds', 'n_training_trials',
      'n_predicted_trials', 'source']),
    ('verification_checks.csv', checks, ['check', 'status']),
]
for name, data, fields in specifications:
    write_csv(OUT / name, data, fields)

summary = {
    'status': 'PASS' if all(r['status'] == 'PASS' for r in checks) else 'REVIEW_REQUIRED',
    'experiment': 'E4.2 — regularisation and training-participant sufficiency',
    'source_hdf5': H5.name, 'source_sha256': source_sha,
    'reference_zip': REFERENCE.name, 'reference_zip_sha256': reference_sha,
    'reference_notebook_04_code_sha256': ref_summary['notebook_code_sha256'],
    'binding_subset_manifest': SUBSETS.name,
    'binding_subset_manifest_sha256': subset_sha,
    'primary_target': 'stored corrected trial-level metcost_mean, net W/kg',
    'population': '284 unrestricted trials with >=100 retained cycles, 21 participants',
    'evaluation_status': 'internally held-out; previously explored Baček outer folds',
    'representation': 'D-ALL9, 54 trial-mean waveform descriptors',
    'model': 'StandardScaler -> Ridge(alpha selected by pooled 4-fold group validation)',
    'alpha_grid': list(ALPHAS),
    'subset_budgets': ['8', '12', 'full (16–17)'],
    'repetitions_per_small_budget_per_outer_fold': 5,
    'subset_assignments': 'binding CSV; exact participant IDs used (no resampling)',
    'outer_test_population': 'the same 284 primary trials for every budget and repeat',
    'inner_splitter_small_subsets': 'sklearn GroupKFold(n_splits=4), shuffle=False',
    'full_development': 'D-ALL9_FIXED reference from Notebook 04, no new fitting',
    'small_subset_fits': 50,
    'inner_candidate_fits_new': 50 * 6 * 4,
    'complete_prediction_records': len(subset_predictions),
    'metric_unit': 'W/kg',
    'training_rmse_warning': 'inner training values are in-sample, not external test estimates',
    'repeat_warning': 'five overlapping subsets; descriptive only, not independent replicates/CI',
    'bootstrap_included': False,
    'started_at_utc': started_at,
    'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    'elapsed_seconds': elapsed_seconds,
    'software': {'python': platform.python_version(), 'numpy': np.__version__,
                 'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__,
                 'h5py': h5py.__version__, 'os': platform.platform(),
                 'cpu_count': os.cpu_count(), 'kernel_python_executable': sys.executable},
    'notebook_filename': NOTEBOOK.name,
    'notebook_code_sha256': notebook_code_sha,
    'code_fingerprint_basis': 'concatenated code-cell sources, UTF-8',
    'verification': {r['check']: r['status'] for r in checks},
}
(OUT / 'run_summary.json').write_text(
    json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for name, *_ in specifications:
        archive.write(OUT / name, arcname=name)
    archive.write(OUT / 'run_summary.json', arcname='run_summary.json')
print('\nCreated', ARCHIVE, '| files:', len(specifications) + 1,
      '| verification:', summary['status'])
